# Day 31: LangGraph Tool Agent

We've built tool agents twice — manually on Day 18, with LangChain on Day 26.

Today we see what was happening under the hood: **the graph**.

## Install

In [ ]:
%pip install langgraph langchain-google-genai langchain --quiet

## Setup

In [ ]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.tools import tool
import requests

load_dotenv(dotenv_path='../.env')
os.environ["GOOGLE_API_KEY"] = os.environ["GEMINI_API_KEY2"]

model = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

HEADERS = {"User-Agent": "genai-50-days-demo/1.0"}

@tool
def get_weather(city: str) -> str:
    """Get current weather for a city."""
    resp = requests.get(f"https://wttr.in/{city}?format=j1", headers=HEADERS)
    data = resp.json()["current_condition"][0]
    return f"{city}: {data['temp_C']}°C, {data['weatherDesc'][0]['value']}"

tools = [get_weather]
print("✅ Model and tool ready")

## Build Graph

`bind_tools` teaches the model about tools. `should_continue` routes: tool calls loop back, no calls end.

In [ ]:
from typing import Annotated, Literal
from typing_extensions import TypedDict
from langchain_core.messages import ToolMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

class State(TypedDict):
    messages: Annotated[list, add_messages]

model_with_tools = model.bind_tools(tools)

def llm_node(state: State):
    return {"messages": [model_with_tools.invoke(state["messages"])]}

def tool_node(state: State):
    results = []
    for call in state["messages"][-1].tool_calls:
        fn = {t.name: t for t in tools}[call["name"]]
        output = fn.invoke(call["args"])
        results.append(ToolMessage(content=str(output), tool_call_id=call["id"]))
    return {"messages": results}

In [ ]:
def should_continue(state: State) -> Literal["tool_node", "__end__"]:
    return "tool_node" if state["messages"][-1].tool_calls else END

builder = StateGraph(State)
builder.add_node("llm_node", llm_node)
builder.add_node("tool_node", tool_node)
builder.add_edge(START, "llm_node")
builder.add_conditional_edges("llm_node", should_continue, ["tool_node", END])
builder.add_edge("tool_node", "llm_node")
agent = builder.compile()
print("✅ Tool agent compiled")

In [ ]:
from IPython.display import Image, display

display(Image(agent.get_graph().draw_mermaid_png()))

## Test: Tool Call

In [ ]:
result = agent.invoke({"messages": [{"role": "user", "content": "What's the weather in Berlin?"}]})

for msg in result["messages"]:
    msg.pretty_print()

## Add Memory

Same graph, one extra parameter. Same `thread_id` = same conversation.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

agent_with_memory = builder.compile(checkpointer=InMemorySaver())
config = {"configurable": {"thread_id": "demo-1"}}

result = agent_with_memory.invoke(
    {"messages": [{"role": "user", "content": "What's the weather in Tokyo?"}]}, config
)
print("Turn 1:", result["messages"][-1].content)

In [ ]:
result = agent_with_memory.invoke(
    {"messages": [{"role": "user", "content": "Is that warm?"}]}, config
)
print("Turn 2:", result["messages"][-1].content)

## Key Takeaways

1. **`bind_tools` + `should_continue`** — the agent decides when to call tools, the graph routes the loop
2. **`add_conditional_edges`** is the core decision point — tool calls loop back, no calls end the graph
3. **`checkpointer=InMemorySaver()`** — one parameter adds persistent memory across turns